## Load information 
### Gather solar radiation data from multiple files

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd
import glob
import os

from shapely.geometry import Point
from helper import *

path = 'data/radiation/'
files = glob.glob(os.path.join(path, '*.csv'))

radiation_data = []
for f in files:
    # Read CSV skipping first two lines and using the third line as header
    df = pd.read_csv(f, skiprows=2)
    # Extract lat/lon from filename (e.g., xxxxxxx_41.88_-87.63_yyyy.csv)
    lat, lon = map(float, os.path.basename(f).split('_')[1:3])
    df['lat'] = lat
    df['lon'] = lon
    df['geometry'] = Point(lon, lat)
    radiation_data.append(df)

radiation_df = pd.concat(radiation_data, ignore_index=True)

Convert stadard dataframe into geopandas dataframe. This will still have multiple timestamps (year, month, day, hour) per (DHI, DNI, GHI) triplet.

In [3]:
import geopandas as gpd

radiation_gdf = gpd.GeoDataFrame(radiation_df, geometry='geometry', crs='EPSG:4326')
radiation_gdf

,Year,Month,Day,Hour,Minute,DHI,DNI,GHI,lat,lon,geometry
0,2024,1,1,0,0,0,0,0,41.9,-87.67,POINT (-87.67 41.9)
1,2024,1,1,1,0,0,0,0,41.9,-87.67,POINT (-87.67 41.9)
2,2024,1,1,2,0,0,0,0,41.9,-87.67,POINT (-87.67 41.9)
3,2024,1,1,3,0,0,0,0,41.9,-87.67,POINT (-87.67 41.9)
4,2024,1,1,4,0,0,0,0,41.9,-87.67,POINT (-87.67 41.9)
...,...,...,...,...,...,...,...,...,...,...,...
1970995,2024,12,31,19,0,6,0,6,42.0,-87.69,POINT (-87.69 42)
1970996,2024,12,31,20,0,11,0,11,42.0,-87.69,POINT (-87.69 42)
1970997,2024,12,31,21,0,83,9,85,42.0,-87.69,POINT (-87.69 42)
1970998,2024,12,31,22,0,26,261,44,42.0,-87.69,POINT (-87.69 42)


In [4]:
print(radiation_gdf.crs)

EPSG:4326


To have a single geopandas data frame with a single triplet per coordinate which will be the year average.

In [5]:
# not removing zeros because those were measurements, not nan
sum_gdf = radiation_gdf.groupby(['lat', 'lon']).agg({
    "GHI": ["mean", "sum"],
    "DNI": ["mean", "sum"],
    "DHI": ["mean", "sum"]
}) 

# Flatten the MultiIndex columns
sum_gdf.columns = ['_'.join(col).strip() for col in sum_gdf.columns.values]

# Reset index to make it easier to join back later
sum_gdf = sum_gdf.reset_index()

sum_gdf['geometry'] = sum_gdf.apply(lambda row: Point(row['lon'], row['lat']), axis=1)
sum_gdf['GHI/DNI'] = sum_gdf['GHI_mean']/sum_gdf['DNI_mean']
sum_gdf = gpd.GeoDataFrame(sum_gdf, geometry='geometry', crs='EPSG:4326') 
sum_gdf = sum_gdf.to_crs('EPSG:3435')  # Chicago's local CRS
sum_gdf

,lat,lon,GHI_mean,GHI_sum,DNI_mean,DNI_sum,DHI_mean,DHI_sum,geometry,GHI/DNI
0,41.76,-87.83,176.532534,1546425,204.086530,1787798,61.949543,542678,POINT (1121575.499 1855536.631),0.864989
1,41.76,-87.81,176.146804,1543046,202.270548,1771890,62.837785,550459,POINT (1127032.158 1855569.192),0.870848
2,41.76,-87.79,175.821233,1540194,202.121918,1770588,62.361872,546290,POINT (1132488.819 1855603.022),0.869877
3,41.76,-87.77,175.758790,1539647,202.185160,1771142,62.280251,545575,POINT (1137945.483 1855638.12),0.869296
4,41.76,-87.75,176.322831,1544588,202.780023,1776353,62.183447,544727,POINT (1143402.149 1855674.487),0.869528
...,...,...,...,...,...,...,...,...,...,...
220,42.04,-87.63,163.377854,1431190,157.358105,1378457,73.096005,640321,POINT (1175305.695 1957951.466),1.038255
221,42.04,-87.61,166.662215,1459961,175.347603,1536045,65.318379,572189,POINT (1180738.588 1957996.763),0.950468
222,42.04,-87.59,167.625342,1468398,176.509703,1546225,65.379680,572726,POINT (1186171.484 1958043.331),0.949666
223,42.04,-87.57,167.549201,1467731,176.476027,1545930,65.348059,572449,POINT (1191604.382 1958091.168),0.949416


### Load building footprints

In [6]:
buildings_gdf = gpd.read_file('data/footprints/buildings.shp')

In [7]:
print('buildings gdf crs: ', buildings_gdf.crs)
print('buildings gdf shape: ', buildings_gdf.shape)
print(buildings_gdf.head())

buildings gdf crs:  EPSG:3435
buildings gdf shape:  (820606, 43)
   BLDG_ID CDB_CITY_I BLDG_STATU  F_ADD1  T_ADD1 PRE_DIR1  ST_NAME1 ST_TYPE1  \
0   729719       None     ACTIVE   10203   10203        S      WOOD       ST   
1    46828       None     ACTIVE    5901    5901        N  KILBOURN      AVE   
2   244234       None     ACTIVE    2830    2830        N  NORMANDY      AVE   
3   451013       None     ACTIVE       0       0     None      None     None   
4   601329       None     ACTIVE       0       0     None      None     None   

  UNIT_NAME NON_STANDA  ... YEAR_BUILT BLDG_SQ_FO BLDG_CONDI  CONDITION_  \
0      None       None  ...       1899     2196.0      SOUND  2003-03-01   
1      None       None  ...       1937     1927.0      SOUND  2003-05-01   
2      None       None  ...       1957     1583.0      SOUND  2002-01-01   
3      None       None  ...          0        0.0       None         NaT   
4      None       None  ...          0        0.0       None         NaT  

## Match radiation points to building footprints

In [8]:
joined_gdf = buildings_gdf.sjoin_nearest(sum_gdf, how='left', distance_col='distance_meters')

In [9]:
# check if the join worked properly and see how far off are the interpolations
(joined_gdf.sort_values(by='distance_meters', ascending=True)).tail(25)

,BLDG_ID,CDB_CITY_I,BLDG_STATU,F_ADD1,T_ADD1,PRE_DIR1,ST_NAME1,ST_TYPE1,UNIT_NAME,NON_STANDA,...,lat,lon,GHI_mean,GHI_sum,DNI_mean,DNI_sum,DHI_mean,DHI_sum,GHI/DNI,distance_meters
720433,812095,None,ACTIVE,0,0,None,None,None,None,None,...,41.76,-87.61,174.367237,1527457.0,196.839840,1724317.0,62.367694,546341.0,0.885833,41924.377372
276391,812104,None,ACTIVE,0,0,None,None,None,None,None,...,41.76,-87.61,174.367237,1527457.0,196.839840,1724317.0,62.367694,546341.0,0.885833,41929.045297
103976,812093,None,ACTIVE,13780,13780,S,LEYDEN,AVE,None,None,...,41.76,-87.61,174.367237,1527457.0,196.839840,1724317.0,62.367694,546341.0,0.885833,41929.387026
98795,812078,None,ACTIVE,956,956,E,138TH,ST,None,None,...,41.76,-87.59,174.525799,1528846.0,197.160388,1727125.0,62.195091,544829.0,0.885197,41949.248104
271634,812129,None,ACTIVE,0,0,None,None,None,None,None,...,41.76,-87.61,174.367237,1527457.0,196.839840,1724317.0,62.367694,546341.0,0.885833,41953.089721
443618,812102,None,ACTIVE,700,700,E,138TH,ST,None,None,...,41.76,-87.61,174.367237,1527457.0,196.839840,1724317.0,62.367694,546341.0,0.885833,41955.550271
316526,812052,None,ACTIVE,0,0,None,None,None,None,None,...,41.76,-87.55,170.833904,1496505.0,195.490183,1712494.0,60.299886,528227.0,0.873875,41955.925346
465291,812137,None,ACTIVE,320,320,E,138TH,ST,None,None,...,41.76,-87.61,174.367237,1527457.0,196.839840,1724317.0,62.367694,546341.0,0.885833,41956.765892
509519,812139,None,ACTIVE,304,304,E,138TH,ST,None,None,...,41.76,-87.61,174.367237,1527457.0,196.839840,1724317.0,62.367694,546341.0,0.885833,41960.177751
765193,812143,None,ACTIVE,324,324,E,138TH,ST,None,None,...,41.76,-87.61,174.367237,1527457.0,196.839840,1724317.0,62.367694,546341.0,0.885833,41960.843871


Compute surface area in $m^2$ from geometry

In [10]:
# Convert to projected CRS (e.g., for Chicago, use EPSG:26971)
joined_gdf = joined_gdf.to_crs(epsg=26916)

# Compute surface area in m²
joined_gdf["surface_area"] = joined_gdf.geometry.area

# Convert back to lat/lon
joined_gdf = joined_gdf.to_crs(epsg=3435)

In [11]:
# Drop rows with missing geometry
joined_gdf = joined_gdf.dropna(subset=["geometry"])

# Drop geometries that are empty
joined_gdf = joined_gdf[~joined_gdf.geometry.is_empty]

# Drop rows with missing building IDs
joined_gdf = joined_gdf.dropna(subset=["BLDG_ID"])

# Reset index to avoid ghosts
joined_gdf = joined_gdf.reset_index(drop=True)

In [12]:
# check for null or empty geometries
print("Null geometries:", joined_gdf.geometry.isna().sum())
print("Empty geometries:", joined_gdf.geometry.apply(lambda g: g.is_empty if g else False).sum())

# inspect problematic rows
joined_gdf[joined_gdf.geometry.isna() | joined_gdf.geometry.apply(lambda g: g.is_empty if g else False)].head()

Null geometries: 0
Empty geometries: 0


,BLDG_ID,CDB_CITY_I,BLDG_STATU,F_ADD1,T_ADD1,PRE_DIR1,ST_NAME1,ST_TYPE1,UNIT_NAME,NON_STANDA,...,lon,GHI_mean,GHI_sum,DNI_mean,DNI_sum,DHI_mean,DHI_sum,GHI/DNI,distance_meters,surface_area


## Solar potential

To first order, 

$\text{kWh = GHI × area × efficiency / 100}$

so we can already compute the annual solar energy output at each building given that

- We have the column GHI_sum (Wh/m²/year)
- We have the column SHAPE_AREA which is the roof surface area in m²
- We can assume that the system efficiency is ~15%

In [13]:
# trim data to avoid slowness during visualizations
gdf = joined_gdf.copy()
gdf = gdf[['GHI_sum', 'GHI/DNI', 'lon', 'lat', 'BLDG_ID', 'surface_area', 'geometry']]
gdf.columns = [col.lower() for col in gdf.columns]
print(gdf.tail())

          ghi_sum   ghi/dni    lon    lat  bldg_id  surface_area  \
820595  1509390.0  0.895395 -87.63  41.82   892539    454.382897   
820596  1509390.0  0.895395 -87.63  41.82   892567    221.633161   
820597  1529009.0  0.886370 -87.61  41.78   892862   4221.451970   
820598  1498887.0  0.894363 -87.67  42.00   892895    137.672129   
820599  1541404.0  0.864921 -87.69  41.76   893374    304.753312   

                                                 geometry  
820595  POLYGON ((1176594.332 1878222.96, 1176593.68 1...  
820596  POLYGON ((1176764.908 1876198.697, 1176740.801...  
820597  POLYGON ((1182951.38 1865095.815, 1182955.443 ...  
820598  POLYGON ((1162617.367 1946605.265, 1162595.09 ...  
820599  POLYGON ((1160141.474 1834136.742, 1160142.582...  


In [14]:
from helper import compute_orientation_angle, categorize_orientation_from_angle
# compute orientation of the longest edge as a proxy for roof orientation (azimuth)
# gdf["orientation_angle"] = gdf.geometry.apply(get_orientation_angle)
gdf = compute_orientation_angle(gdf)    

# classify orientation (cardinal direction)
# gdf["orientation"] = gdf["orientation_angle"].apply(categorize_orientation)
gdf = categorize_orientation_from_angle(gdf)

[0 4 4 ... 2 6 4]
['North' 'Northeast' 'East' 'Southeast' 'South' 'Southwest' 'West'
 'Northwest' 'North']
['North' 'South' 'South' ... 'East' 'West' 'South']
     ghi_sum   ghi/dni    lon    lat  bldg_id  surface_area  \
0  1548665.0  0.859872 -87.67  41.76   729719    139.830836   
1  1536622.0  0.859507 -87.75  41.98    46828    105.659919   
2  1533574.0  0.864815 -87.79  41.94   244234    162.665625   
3  1538370.0  0.864420 -87.73  41.80   451013     36.624010   
4  1548665.0  0.859872 -87.67  41.76   601329     45.128830   

                                            geometry  orientation  \
0  POLYGON ((1166267.643 1836882.539, 1166230.143...     0.572939   
1  POLYGON ((1145406.643 1938862.039, 1145406.143...   181.397181   
2  POLYGON ((1131149.143 1918209.539, 1131148.643...   181.385918   
3  POLYGON ((1147963.143 1872636.539, 1147962.143...   182.726311   
4  POLYGON ((1165804.143 1854234.039, 1165804.143...   271.193489   

  orientation_cat  
0           North  
1      

In [15]:
print(gdf.columns.tolist())

['ghi_sum', 'ghi/dni', 'lon', 'lat', 'bldg_id', 'surface_area', 'geometry', 'orientation', 'orientation_cat']


In [16]:
# estimate kwh per year
# gdf = gdf.apply(get_kwh, axis=1)
gdf = compute_kwh(gdf)
print(gdf['orientation_cat'].head())

0    North
1    South
2    South
3    South
4     West
Name: orientation_cat, dtype: object


## 🌍 Climate Impact

We can also estimate avoided CO₂ emissions by replacing grid electricity with clean solar.

CO₂ Avoided per Year (kg):

$\text{CO}_2 \text{Avoided = Annual Energy Output (kWh)} \times \text{Grid Emission Factor (kgCO}_2\text{/kWh)}$

A typical U.S. emission factor is ~0.4–0.5 kgCO₂/kWh, but this varies by region. Source specific kgCO₂/kWh are typically 0.90 for coal, 0.40 gas, ~0.05 for nuclear.


In [17]:
# estimate avoided CO2 per year
gdf["co2_avoided_t"] = gdf.apply(
    lambda row: get_co2_avoided(row["kwh_estimate"]),
    axis=1
)

## 💰 Financial Metrics

To understand the economic feasibility of rooftop solar, we compute basic financial indicators using standard assumptions for installation costs and electricity prices.

* System Size (kW):
We estimate installed capacity from roof area using a typical power density of ~0.2 kW/m².
* CAPEX (Capital Expenditure):
Initial cost of installing solar panels, estimated as:

$\text{CAPEX=System Size (kW)×Cost per kW}$

A common assumption is \$1000–\$1500/kW installed.

* Annual Savings:
Money saved by offsetting electricity consumption, calculated as:


$\text{Savings=Annual Energy Output (kWh)}\times\text{Electricity Price (\$/kWh)}$

* Simple Payback Period (years):
How long until the installation pays for itself:


$\text{Payback}=\frac{\text{CAPEX}}{\text{Annual Savings}}$

* ROI (Return on Investment):
Basic measure of profitability:

$\text{ROI}=\frac{\text{Annual Savings}}{\text{CAPEX}}$

In [18]:
gdf_finance = get_finance_computations(gdf)
gdf_finance.head()

,ghi_sum,ghi/dni,lon,lat,bldg_id,surface_area,geometry,orientation,orientation_cat,raw_kwh_estimate,...,kwh_estimate,co2_avoided_t,annual_kwh,system_kw,capex_per_kw_adj,capex_usd,annual_savings_usd,annual_om_usd,simple_payback_years,simple_roi
0,1548665.0,0.859872,-87.67,41.76,729719,139.830836,"POLYGON ((1166267.643 1836882.539, 1166230.143...",0.572939,North,22510.489096,...,18008.391277,7.203357,18008.391277,18.877163,1390.367235,26246.188737,2701.258692,262.461887,10.761942,0.092920
1,1536622.0,0.859507,-87.75,41.98,46828,105.659919,"POLYGON ((1145406.643 1938862.039, 1145406.143...",181.397181,South,16877.255079,...,16877.255079,6.750902,16877.255079,14.264089,1334.337260,19033.105535,2531.588262,190.331055,8.129438,0.123010
2,1533574.0,0.864815,-87.79,41.94,244234,162.665625,"POLYGON ((1131149.143 1918209.539, 1131148.643...",181.385918,South,25931.343413,...,25931.343413,10.372537,25931.343413,21.959859,1399.348559,30729.497556,3889.701512,307.294976,8.577892,0.116579
3,1538370.0,0.864420,-87.73,41.80,451013,36.624010,"POLYGON ((1147963.143 1872636.539, 1147962.143...",182.726311,South,5856.675874,...,5856.675874,2.342670,5856.675874,4.944241,1615.911240,7989.455168,878.501381,79.894552,10.004241,0.099958
4,1548665.0,0.859872,-87.67,41.76,601329,45.128830,"POLYGON ((1165804.143 1854234.039, 1165804.143...",271.193489,West,7265.007190,...,6683.806615,2.673523,6683.806615,6.092392,1369.826849,8345.522152,1002.570992,83.455222,9.079947,0.110133


In [19]:
# save geodataframe
gdf_finance.to_file('data/solar_summary.geojson', driver='GeoJSON')
gdf_finance.drop(columns="geometry").to_csv("data/solar_summary.csv", index=False)

In [20]:
import shapely
import geopandas
shapely.__version__, geopandas.__version__


('2.1.2', '1.1.1')

Sort buildings from highest to lowest potential energy output and save the top 100.

In [21]:
col_pop = "kwh_estimate"
if col_pop not in gdf_finance.columns:
    gdf_finance.insert(0, col_pop, gdf.pop(col_pop))

# Sort by energy output
gdf_sorted = gdf_finance.sort_values(by="kwh_estimate", ascending=False)

# Take top 100 and bottom 100
top_100 = gdf_sorted.head(100).copy()
top_100["ranking"] = "Top 100"

bottom_100 = gdf_sorted.tail(100).copy()
bottom_100["ranking"] = "Bottom 100"

# Combine into a single GeoDataFrame
top_bottom_100 = pd.concat([top_100, bottom_100])

# save to geojson

top_bottom_100.to_file("data/top_bottom_100_buildings.geojson", driver="GeoJSON")
top_bottom_100.drop(columns="geometry").to_csv("data/top_bottom_100_buildings.csv", index=False)
top_bottom_100.tail()

,ghi_sum,ghi/dni,lon,lat,bldg_id,surface_area,geometry,orientation,orientation_cat,raw_kwh_estimate,...,co2_avoided_t,annual_kwh,system_kw,capex_per_kw_adj,capex_usd,annual_savings_usd,annual_om_usd,simple_payback_years,simple_roi,ranking
513622,1548310.0,0.864427,-87.71,41.76,840667,0.390372,"POLYGON ((1156456.043 1856056.039, 1156456.043...",1.067217e-07,North,62.829163,...,0.020105,50.263331,0.052700,1760.931530,92.801522,7.539500,0.928015,14.036413,0.071243,Bottom 100
335701,1548310.0,0.864427,-87.71,41.76,840551,0.264581,"POLYGON ((1155899.243 1832153.239, 1155899.243...",3.600000e+02,North,42.583434,...,0.013627,34.066747,0.035718,1712.224706,61.157932,5.110012,0.611579,13.595387,0.073554,Bottom 100
66446,1524407.0,0.892595,-87.67,41.86,840559,0.187060,"POLYGON ((1165773.143 1893291.939, 1165773.243...",8.663354e+01,East,29.641925,...,0.010908,27.270571,0.025253,2036.787963,51.435216,4.090586,0.514352,14.382510,0.069529,Bottom 100
562062,1488394.0,0.906252,-87.63,41.86,0,0.175582,"POLYGON ((1175766.159 1894786.91, 1175760.143 ...",5.551739e+00,North,27.165766,...,0.008693,21.732613,0.023704,1795.066801,42.549449,3.259892,0.425494,15.011814,0.066614,Bottom 100
387783,1504999.0,0.896953,-87.67,41.98,871905,0.070580,"POLYGON ((1166147.209 1933153.446, 1166144.645...",1.340421e+01,North,11.041822,...,0.003533,8.833458,0.009528,1888.960288,17.998513,1.325019,0.179985,15.718766,0.063618,Bottom 100


### Visualize location of top 100 candidate buildings

In [22]:
import plotly.express as px
import json
from app.viz import plot_top_k_mapbox

fig_top_k = plot_top_k_mapbox(top_100)
fig_top_k.show()